# finance / household / lumps

The expected annual cost of the big irregular expenses and its 95th percentile.

| Term | In This Notebook |
|---|---|
| **Intent** | Model the lumpy spending that a smooth growth rate hides. Placeholder: five synthetic annual lump totals |
| **Project Flavour** | `static`, local to the notebook (`model.md` 3.2) |
| **Data Splits** | `training` ten synthetic annual totals of irregular spending, until the `bank` table exists; `golden` the exact request that becomes `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, target, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/household/lumps"
MODEL_NAME = "finance_household_lumps"
TARGETS = [{"kind": "local"}]
TRAINING = "household_lumps_training_2026-11-05_2026-11-05"
GOLDEN = "household_lumps_golden"
INPUT = "finance_household_lumps_in"
OUTPUTS = ["finance_household_lumps_expected_annual_dollars", "finance_household_lumps_p95_annual_dollars"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training` file only when it is absent (`model.md` 6.5). The source is synthetic until the real series is loaded, so no query runs.

In [ ]:
training_pl, training_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING,
    lambda: {"year": list(range(2016, 2026)), "lump_dollars": [12000.0, 30000.0, 8000.0, 15000.0, 52000.0, 9000.0, 20000.0, 11000.0, 25000.0, 18000.0]},
)
print("training", training_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** take the mean and the 95th percentile of the annual totals in `training` only (R6). The compound Poisson per category in the plan replaces the empirical percentile once there are enough years.

In [ ]:
import numpy as np

lumps = training_pl["lump_dollars"].to_numpy()
CONSTANTS = dict(zip(OUTPUTS, [round(float(lumps.mean()), 0), round(float(np.percentile(lumps, 95)), 0)], strict=True))
print("constants", CONSTANTS)
assert CONSTANTS[OUTPUTS[0]] == 20000.0 and CONSTANTS[OUTPUTS[1]] > CONSTANTS[OUTPUTS[0]], "the lump summary is not a mean below a high percentile"

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows, build the signature, write a `static` project whose constants are the fitted values, verify it and replay the golden through the library loader (R1, R3).

In [ ]:
from workbench.model import build_signature, package_project, signature_input, signature_output
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUT: [1.0, None], "expect": ["ok", "error"]})

signature = build_signature(
    MODEL_NAME,
    [signature_input(INPUT, "none", "Unused input.")],
    [
        signature_output("finance_household_lumps_expected_annual_dollars", "none", "Expected annual spend on irregular expenses, dollars.", datatype="FP64"),
        signature_output("finance_household_lumps_p95_annual_dollars", "none", "Annual spend on irregular expenses that is exceeded one year in twenty, dollars.", datatype="FP64"),
    ],
)
project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic annual totals of irregular spending, training split",
        [training_path, golden_path],
        "mean and empirical 95th percentile, until a compound Poisson is fitted",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped loader")